In [1]:
from model import Retriever
retriever = Retriever('cambridgeltl/SapBERT-from-PubMedBERT-fulltext')
retriever.load_dictionary_all('./umls_dictionary.txt')
retriever.load_dictionary_bodyloc('./umls_body_loc_dictionary.txt')
retriever.embed_dictionary(32768)
retriever.faiss_setup()

Checking if cache file exists at ./cache//dense_embed_all.pt
Cache file found. Loading dense embeddings from cache.
Checking if cache file exists at ./cache//dense_embed_bodyloc.pt
Cache file found. Loading dense embeddings from cache.


In [55]:
import pandas as pd
import os
import json
annotation = []
path = '../ehrllm/0821_model/structure_eval/annotations/MGB/'
annotation_file = [item for item in os.listdir(path) if not item.startswith('.')]
dfs = []
for fi in annotation_file:
    dfs.append(pd.read_csv(path + fi))
    dfs[-1].rename(columns={'Label': 'label', 'Labels': 'label', 'labels': 'label'}, inplace=True)
dfs = pd.concat(dfs)
annotations_gold = dfs[dfs['label']!=3]
all_code = retriever.embedding_retrieval_all(annotations_gold['phrase'].tolist())
loc_code = retriever.embedding_retrieval_bodyloc([item if not pd.isna(item) else None for item in annotations_gold['body_location'].tolist()])
# all_code = [list(item.keys())[0] if item is not None else None for item in all_code]
# loc_code = [list(item.keys())[0] if item is not None else None for item in loc_code]
annotations_gold['code'] = all_code
annotations_gold['body_location_code'] = loc_code


path = '../ehrllm/0821_model/annotation_ood/'
eval_file = [item for item in os.listdir(path) if not item.startswith('.')]
eval_dfs = []
for fi in eval_file:
    print(fi, path)
    eval_dfs.append(pd.read_csv(path + fi + '/gpt4o_model_result.csv'))
eval_dfs = pd.concat(eval_dfs)

print(eval_dfs[['code', 'mention']])
print(annotations_gold[['code', 'phrase']])
eval_dfs.to_csv('pred.csv')
annotations_gold.to_csv('refs.csv')

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 93.84it/s]

0 ../ehrllm/0821_model/annotation_ood/
19 ../ehrllm/0821_model/annotation_ood/
38 ../ehrllm/0821_model/annotation_ood/
57 ../ehrllm/0821_model/annotation_ood/
                                              code  \
0                             {"C0577559": "mass"}   
1                   {"C0524468": "right shoulder"}   
2                             {"C0030193": "pain"}   
3                             {"C0178298": "skin"}   
4                   {"C0023269": "leiomyosarcoma"}   
..                                             ...   
17  {"C2187547": "retroperitoneal leiomyosarcoma"}   
18       {"C1709157": "negative surgical margins"}   
19                       {"C1522449": "radiation"}   
20                       {"C4331246": "restaging"}   
21               {"C0420333": "follow-up 3 weeks"}   

                          mention  
0                            mass  
1                  right shoulder  
2                            pain  
3                            skin  
4           


/tmp/ipykernel_45846/3944159855.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations_gold['code'] = all_code
/tmp/ipykernel_45846/3944159855.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations_gold['body_location_code'] = loc_code


In [56]:
def eval_recall_precision(annotations_gold, eval_dfs):

    ner_labels = annotations_gold['code'].tolist()
    ner_labels = [list(json.loads(item).keys())[0] for item in ner_labels]
    ner_predic = eval_dfs['code'].tolist()
    ner_predic = [list(json.loads(item).keys())[0] for item in ner_predic if type(item) == str]
    print(ner_labels)
    print(ner_predic)
    
    # ner_correct = 0
    # correct_phrases = []
    # for item in ner_predic:
    #     if item in ner_labels:
    #         ner_correct += 1
    #         ner_labels.remove(item)
    #         correct_phrases.append(item)

    # recall = ner_correct / len(annotations_gold['phrase'])
    # precision = ner_correct / len(eval_dfs['mention'])

    recall = len(set(ner_labels).intersection(ner_predic)) / len(set(ner_labels))
    precision = len(set(ner_labels).intersection(ner_predic)) / len(set(ner_predic))
    print('NER recall:\t', recall)
    print('NER precision:\t', precision)

    return 2 * recall * precision / (precision + recall)
    columns = ['body_location_code']
    df_predic = eval_dfs[eval_dfs['mention'].isin(correct_phrases)]
    df_labels = annotations_gold[annotations_gold['phrase'].isin(correct_phrases)]
    for col in columns:
        labels = [(I+'_'+P).lower() for I, P in zip(df_labels[col].tolist(),df_labels['phrase'].tolist()) if I not in ["not applicable", np.nan, 'null']]
        predic = [(I+'_'+P).lower() for I, P in zip(df_predic[col].tolist(),df_predic['mention'].tolist()) if I not in ["not applicable", np.nan, 'null']]
        label_number = len(labels)
        predi_number = len(predic)
        print('labs', labels)
        print('pred', predic)
        correct = 0
        for item in predic:
            if item in labels:
                correct += 1
                labels.remove(item)
        print(f'{col} recall:\t', correct / (label_number+1e-10), label_number)
        print(f'{col} precision:\t', correct / (predi_number+1e-10), predi_number)

In [57]:
eval_recall_precision(annotations_gold, eval_dfs)

['C0030193', 'C0728940', 'C0023269', 'C1709603', 'C0184918', 'C0023269', 'C1709157', 'C0017168', 'C0040264', 'C1384641', 'C0009376', 'C0007102', 'C1378703', 'C1144661', 'C0020538', 'C0855057', 'C0375983', 'C0024198', 'C0020264', 'C0000970', 'C0003864', 'C0065374', 'C0301532', 'C0039225', 'C0008318', 'C0008318', 'C0991533', 'C0020517', 'C0030842', 'C4316895', 'C0425586', 'C0019214', 'C0080078', 'C0578736', 'C0184898', 'C0034599', 'C1824234', 'C0577559', 'C0202823', 'C0006826', 'C0030664', 'C3163665', 'C0855058', 'C0543478', 'C0030415', 'C0027651', 'C1382187', 'C1382187', 'C0812425', 'C0728940', 'C1337016', 'C4551687', 'C2733303', 'C0677930', 'C0027651', 'C0475445', 'C0475440', 'C1300729', 'C0023269', 'C1261473', 'C0449574', 'C0919553', 'C1709047', 'C1334928', 'C1321236', 'C1261473', 'C0600558', 'C1024573', 'C0021044', 'C4317108', 'C0431085', 'C0011696', 'C0001271', 'C1334660', 'C0184918', 'C1709157', 'C4331246', 'C1522449', 'C5203986']
['C0577559', 'C0524468', 'C0030193', 'C0178298', 'C

0.5223880597014925

In [52]:

files = os.listdir('./mimic_pred/')
files = [item for item in files if item.endswith('csv')]
path = '../ehrllm/0821_model/structure_eval/annotations/'
overall_f1 = []
for fi in files:
    prefix = fi.rstrip('_model_result.csv')
    dfs = pd.read_csv(path + f'{prefix}.csv')
    dfs.rename(columns={'Label': 'label', 'Labels': 'label', 'labels': 'label', 'Unnamed: 9': 'label'}, inplace=True)
    annotations_gold = dfs[dfs['label']!=3]
    all_code = retriever.embedding_retrieval_all(annotations_gold['phrase'].tolist())
    loc_code = retriever.embedding_retrieval_bodyloc([item if not pd.isna(item) else None for item in annotations_gold['body_location'].tolist()])
    annotations_gold['code'] = all_code
    if len(loc_code) == 0:
        loc_code = [None] * len(annotations_gold['code'])
    annotations_gold['body_location_code'] = loc_code
    
    eval_dfs = pd.read_csv('./mimic_pred/' + fi)
    
    overall_f1.append(eval_recall_precision(annotations_gold, eval_dfs))

# path = '../ehrllm/0821_model/annotation_ood/'
# eval_file = [item for item in os.listdir(path) if not item.startswith('.')]
# eval_dfs = []
# for fi in eval_file:
#     print(fi, path)
#     eval_dfs.append(pd.read_csv(path + fi + '/gpt4o_model_result.csv'))
# eval_dfs = pd.concat(eval_dfs)

# print(eval_dfs[['code', 'mention']])
# print(annotations_gold[['code', 'phrase']])
# eval_dfs.to_csv('pred.csv')
# annotations_gold.to_csv('refs.csv')

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 174.99it/s]
/tmp/ipykernel_45846/436760000.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations_gold['code'] = all_code
/tmp/ipykernel_45846/436760000.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations_gold['body_location_code'] = loc_code
100%|███████████████

['C0004339', 'C0240859', 'C0043144', 'C0035508', 'C0019214', 'C0234233', 'C0160950', 'C0013491', 'C0149651', 'C0010520', 'C0013604', 'C0011620', 'C0041582', 'C2004491', 'C0205234', 'C0020614', 'C0918147', 'C2826293', 'C0034065', 'C0039985', 'C0412620', 'C3179130', 'C0879373', 'C3165806']
['C0034642', 'C0043144', 'C0035508', 'C4759928', 'C0019214', 'C0018944', 'C0009938', 'C0011620', 'C0020614', 'C0918147', 'C0918147', 'C0342966', 'C0034065', 'C0039985', 'C0412620', 'C3179130', 'C0879373', 'C3165806']
NER recall:	 0.5
NER precision:	 0.7058823529411765
['C0472567', 'C0034991', 'C0743221', 'C0472567', 'C0009450', 'C0740341', 'C0489941', 'C0489941', 'C0699129', 'C0043031', 'C0525032', 'C0021107', 'C3873974', 'C0525032', 'C0042879', 'C0016709', 'C0525032', 'C0699129', 'C0048470', 'C0242656', 'C0302148', 'C0749975', 'C0016277', 'C0877131', 'C0016277', 'C0042338', 'C0001367', 'C3872095', 'C0451615']
['C1145670', 'C0183993', 'C0184159', 'C3878457', 'C0009450', 'C0740341', 'C0489941', 'C048994

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 198.76it/s]
/tmp/ipykernel_45846/436760000.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations_gold['code'] = all_code
/tmp/ipykernel_45846/436760000.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations_gold['body_location_code'] = loc_code
100%|███████████████

['C1522614', 'C0000731', 'C4085862', 'C0042963', 'C3879329', 'C0589121', 'C0699129', 'C0699129', 'C0699129', 'C0525032', 'C0489553', 'C0723148', 'C0700776', 'C0876064', 'C0282386', 'C0002598', 'C4552779', 'C0286036']
['C0150595', 'C0000731', 'C4085862', 'C0042963', 'C2607832', 'C0043031', 'C0043031', 'C0043031', 'C0525032', 'C0717368', 'C0025859', 'C0907402', 'C0282386', 'C0002598', 'C0000970', 'C0050940']
NER recall:	 0.375
NER precision:	 0.42857142857142855
['C0086787', 'C0005802', 'C0199782', 'C0557977', 'C0392201']
['C1274039', 'C0717368', 'C0428554', 'C1145751', 'C0557977', 'C0392201']
NER recall:	 0.4
NER precision:	 0.3333333333333333
['C0042963', 'C4084776', 'C4284399', 'C0281936', 'C1258215', 'C0412620', 'C0599156', 'C0677554', 'C0267771', 'C1322279', 'C0400847', 'C0801760', 'C0743973', 'C0700292', 'C0085559', 'C0003232', 'C0566075', 'C0009378', 'C2348535', 'C0235329', 'C0028778', 'C0011414', 'C0085704', 'C0032285', 'C0002915', 'C0021925', 'C0009378', 'C2348535', 'C0179740', 

ZeroDivisionError: float division by zero

In [53]:
import numpy as np
np.mean(overall_f1)

0.4910152883809675

In [54]:
overall_f1

[0.5853658536585366,
 0.6976744186046512,
 0.41379310344827586,
 0.4210526315789474,
 0.5263157894736842,
 0.39999999999999997,
 0.3636363636363636,
 0.6779661016949152,
 0.3333333333333333]